# 04 — Analyze a long mixed-workout video

Upload the `calorie_tcn_report.zip` produced by notebook 03 and one workout video. This notebook extracts pose landmarks at 15 FPS, classifies overlapping 64-frame windows, smooths the probabilities, rejects uncertain predictions as `unknown`, merges adjacent predictions, and exports a timeline.

This stage recognizes **what exercise happens when**. Repetition counting and calorie calculation are separate modules that will consume the resulting segments.


In [ ]:
%pip install -q mediapipe==0.10.35

from pathlib import Path
import hashlib
import json
import os
import shutil
import urllib.request
import zipfile

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import pandas as pd
import torch
from google.colab import files
from torch import nn
from tqdm.auto import tqdm

TARGET_FPS = 15
WINDOW = 64
STRIDE = 16
SMOOTH_WINDOWS = 5
CONFIDENCE_THRESHOLD = 0.55
MIN_SEGMENT_SECONDS = 2.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("MediaPipe:", mp.__version__)


In [ ]:
print("Upload calorie_tcn_report.zip")
model_upload = files.upload()
model_zip = next((name for name in model_upload if name.lower().endswith(".zip")), None)
assert model_zip, "Upload the report ZIP made by notebook 03."

MODEL_ROOT = Path("/content/tcn_report")
if MODEL_ROOT.exists():
    shutil.rmtree(MODEL_ROOT)
MODEL_ROOT.mkdir()
with zipfile.ZipFile(model_zip) as archive:
    archive.extractall(MODEL_ROOT)

MODEL_FILE = next(MODEL_ROOT.rglob("tcn_model.pt"), None)
assert MODEL_FILE, "tcn_model.pt was not found in the report ZIP."
print("Model:", MODEL_FILE)


In [ ]:
# Default external smoke-test video. Set VIDEO_URL = "" to upload your own file.
VIDEO_URL = (
    "https://commons.wikimedia.org/wiki/"
    "Special:Redirect/file/Jumping_jacks_and_burpees.webm"
)
VIDEO_LICENSE = "CC BY-SA 4.0"
VIDEO_SOURCE_PAGE = "https://commons.wikimedia.org/wiki/File:Jumping_jacks_and_burpees.webm"

if VIDEO_URL:
    VIDEO_PATH = Path("/content/jumping_jacks_and_burpees.webm")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Downloaded external test video:", VIDEO_PATH)
else:
    print("Upload one MP4, MOV, or WebM workout video")
    video_upload = files.upload()
    VIDEO_PATH = next(
        (Path(name) for name in video_upload if Path(name).suffix.lower() in {".mp4", ".mov", ".webm"}),
        None,
    )
    assert VIDEO_PATH, "No supported video was uploaded."
    VIDEO_LICENSE = "user-provided"
    VIDEO_SOURCE_PAGE = "local upload"

print("Video:", VIDEO_PATH)
print("Size MB:", round(VIDEO_PATH.stat().st_size / 1024**2, 2))


In [ ]:
MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"
POSE_MODEL_PATH = Path("/content/pose_landmarker_full.task")
if not POSE_MODEL_PATH.exists():
    urllib.request.urlretrieve(MODEL_URL, POSE_MODEL_PATH)
print("Pose model ready:", POSE_MODEL_PATH.exists())


## Load the TCN

The architecture must match training. The saved checkpoint supplies the learned weights and the exact class order, so the notebook also works after adding `unknown`.


In [ ]:
class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.2):
        super().__init__()
        padding = dilation
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(out_channels, out_channels, 3, padding=padding, dilation=dilation),
            nn.BatchNorm1d(out_channels), nn.ReLU(), nn.Dropout(dropout),
        )
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.residual(x))


class SmallTCN(nn.Module):
    def __init__(self, input_features, num_classes):
        super().__init__()
        channels = [64, 96, 128, 128]
        blocks, current = [], input_features
        for level, output in enumerate(channels):
            blocks.append(TemporalBlock(current, output, dilation=2 ** level))
            current = output
        self.features = nn.Sequential(*blocks)
        self.classifier = nn.Linear(current, num_classes)

    def forward(self, x):
        return self.classifier(self.features(x).mean(dim=-1))


checkpoint = torch.load(MODEL_FILE, map_location=DEVICE, weights_only=False)
CLASSES = list(checkpoint["classes"])
WINDOW = int(checkpoint.get("window_frames", WINDOW))
model = SmallTCN(int(checkpoint.get("input_features", 34)), len(CLASSES)).to(DEVICE)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print("Classes:", CLASSES)
print("Window:", WINDOW, "frames =", round(WINDOW / TARGET_FPS, 2), "seconds")


## Extract pose landmarks

Each sampled frame becomes 17 normalized body points. Missing detections are interpolated. The whole ten-minute video is only about 9,000 landmark frames at 15 FPS, so the compact array fits comfortably in memory.


In [ ]:
KEEP_17 = [0, 2, 5, 7, 8, 11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
LEFT_SHOULDER, RIGHT_SHOULDER = 5, 6
LEFT_HIP, RIGHT_HIP = 11, 12


def normalize_pose(frame):
    xyz = frame[:, :3].copy()
    hip_center = (xyz[LEFT_HIP] + xyz[RIGHT_HIP]) / 2.0
    shoulder_center = (xyz[LEFT_SHOULDER] + xyz[RIGHT_SHOULDER]) / 2.0
    scale = max(
        float(np.linalg.norm(xyz[LEFT_SHOULDER] - xyz[RIGHT_SHOULDER])),
        float(np.linalg.norm(shoulder_center - hip_center)),
        1e-6,
    )
    xyz = (xyz - hip_center) / scale
    return np.concatenate([xyz, frame[:, 3:4]], axis=1)


def fill_missing(sequence):
    flat = sequence.reshape(len(sequence), -1)
    filled = pd.DataFrame(flat).interpolate(limit_direction="both").fillna(0).to_numpy()
    return filled.reshape(sequence.shape).astype(np.float32)


cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "OpenCV could not open the video."
source_fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
source_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
duration_s = source_frames / source_fps if source_frames else 0
sample_interval = 1.0 / TARGET_FPS
next_sample_s = 0.0
frame_index = 0
poses, times, detected = [], [], 0

options = mp.tasks.vision.PoseLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(POSE_MODEL_PATH)),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    output_segmentation_masks=False,
)

progress = tqdm(total=source_frames if source_frames else None, desc="Reading video frames")
with mp.tasks.vision.PoseLandmarker.create_from_options(options) as landmarker:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        time_s = frame_index / source_fps
        frame_index += 1
        progress.update(1)
        if time_s + 1e-9 < next_sample_s:
            continue
        next_sample_s += sample_interval
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb),
            int(round(time_s * 1000)),
        )
        if result.pose_landmarks:
            landmarks = result.pose_landmarks[0]
            pose = np.asarray([
                [landmarks[i].x, landmarks[i].y, landmarks[i].z, float(landmarks[i].visibility or 0.0)]
                for i in KEEP_17
            ], dtype=np.float32)
            poses.append(normalize_pose(pose))
            detected += 1
        else:
            poses.append(np.full((17, 4), np.nan, dtype=np.float32))
        times.append(time_s)

progress.close()
cap.release()
assert poses, "No frames were sampled from the video."
sequence = fill_missing(np.stack(poses))
detection_rate = detected / len(sequence)
print("Duration seconds:", round(duration_s, 1))
print("Sampled landmark frames:", len(sequence))
print("Pose detection rate:", round(detection_rate, 3))
assert detection_rate >= 0.5, "Pose detection is too low; use a clearer full-body video."


## Sliding-window predictions and smoothing

The model sees about 4.3 seconds at a time. We move the window by roughly one second, average five neighboring probability vectors, and apply a confidence threshold. This removes much of the frame-to-frame flicker.


In [ ]:
starts = list(range(0, max(1, len(sequence) - WINDOW + 1), STRIDE))
if len(sequence) > WINDOW and starts[-1] != len(sequence) - WINDOW:
    starts.append(len(sequence) - WINDOW)

features, centers = [], []
for start in starts:
    window = sequence[start:start + WINDOW]
    if len(window) < WINDOW:
        window = np.concatenate([window, np.repeat(window[-1:], WINDOW - len(window), axis=0)])
    features.append(window[:, :, :2].reshape(WINDOW, -1).T)
    centers.append(min((start + WINDOW / 2) / TARGET_FPS, times[-1]))

batch = torch.from_numpy(np.asarray(features, dtype=np.float32))
probabilities = []
with torch.no_grad():
    for start in range(0, len(batch), 256):
        probabilities.append(model(batch[start:start + 256].to(DEVICE)).softmax(1).cpu().numpy())
probabilities = np.concatenate(probabilities)

smoothed = pd.DataFrame(probabilities).rolling(
    SMOOTH_WINDOWS, center=True, min_periods=1
).mean().to_numpy()
predicted_ids = smoothed.argmax(axis=1)
confidences = smoothed.max(axis=1)
predicted_labels = np.asarray([CLASSES[i] for i in predicted_ids], dtype=object)
predicted_labels[confidences < CONFIDENCE_THRESHOLD] = "unknown"

window_predictions = pd.DataFrame({
    "center_s": centers,
    "predicted_class": predicted_labels,
    "confidence": confidences,
})
display(window_predictions.head(10))


In [ ]:
def build_segments(predictions, video_end_s):
    centers = predictions["center_s"].to_numpy(dtype=float)
    boundaries = np.empty(len(centers) + 1, dtype=float)
    boundaries[0] = 0.0
    boundaries[-1] = video_end_s
    if len(centers) > 1:
        boundaries[1:-1] = (centers[:-1] + centers[1:]) / 2.0

    raw = []
    for index, row in enumerate(predictions.itertuples(index=False)):
        start_s = boundaries[index]
        end_s = boundaries[index + 1]
        if raw and raw[-1]["label"] == row.predicted_class:
            raw[-1]["end_s"] = end_s
            raw[-1]["confidences"].append(row.confidence)
        else:
            raw.append({
                "label": row.predicted_class,
                "start_s": start_s,
                "end_s": end_s,
                "confidences": [row.confidence],
            })

    # Short isolated segments are treated as uncertain transition noise.
    for segment in raw:
        if segment["end_s"] - segment["start_s"] < MIN_SEGMENT_SECONDS:
            segment["label"] = "unknown"

    merged = []
    for segment in raw:
        if merged and merged[-1]["label"] == segment["label"]:
            merged[-1]["end_s"] = segment["end_s"]
            merged[-1]["confidences"].extend(segment["confidences"])
        else:
            merged.append(segment)

    return pd.DataFrame([{
        "start_s": round(item["start_s"], 2),
        "end_s": round(item["end_s"], 2),
        "duration_s": round(item["end_s"] - item["start_s"], 2),
        "predicted_class": item["label"],
        "mean_confidence": round(float(np.mean(item["confidences"])), 3),
    } for item in merged])


timeline = build_segments(window_predictions, times[-1])
display(timeline)
print("Segments:", len(timeline))


In [ ]:
OUTPUT_ROOT = Path("/content/mixed_video_analysis")
OUTPUT_ROOT.mkdir(exist_ok=True)
window_predictions.to_csv(OUTPUT_ROOT / "window_predictions.csv", index=False)
timeline.to_csv(OUTPUT_ROOT / "timeline.csv", index=False)

plt.figure(figsize=(16, 5))
for class_index, class_name in enumerate(CLASSES):
    plt.plot(centers, smoothed[:, class_index], label=class_name, linewidth=1)
plt.axhline(CONFIDENCE_THRESHOLD, color="black", linestyle="--", linewidth=1, label="threshold")
plt.xlabel("Video time (seconds)")
plt.ylabel("Smoothed probability")
plt.title("Mixed-workout TCN predictions")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "probability_timeline.png", dpi=170)
plt.show()

video_sha256 = hashlib.sha256(VIDEO_PATH.read_bytes()).hexdigest()
summary = {
    "video": VIDEO_PATH.name,
    "video_source": VIDEO_SOURCE_PAGE,
    "video_license": VIDEO_LICENSE,
    "video_sha256": video_sha256,
    "duration_s": float(times[-1]),
    "pose_detection_rate": float(detection_rate),
    "classes": CLASSES,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "segments": len(timeline),
}
(OUTPUT_ROOT / "summary.json").write_text(json.dumps(summary, indent=2))
zip_path = shutil.make_archive("/content/mixed_video_analysis", "zip", root_dir=OUTPUT_ROOT)
print("Analysis ZIP:", zip_path)
files.download(zip_path)
